# Fine-tuning template for gemma3

First the model is fine-tunied to instruction dataset. Then RL is applied to optimize it for tool calling.

Refs:
- [fine tuning gemma](https://unsloth.ai/docs/models/gemma-3-how-to-run-and-fine-tune#fine-tuning-gemma-3-in-unsloth)
- [rl training](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide/tutorial-train-your-own-reasoning-model-with-grpo)

## Instruction fine-tuning

### Environment setup

In [ ]:
import os, re
pip_args = ["-qqq --extra-index-url https://download.pytorch.org/whl/cu130", "torch==2.10.0", "torchvision==0.25.0+cu130", "torchaudio==2.10.0+cu130", "unsloth", "transformers==4.56.2", "tf_keras"] # "vllm"
if "COLAB_" in "".join(os.environ.keys()):
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    pip_args.append(xformers)
    pip_args.extend(["sentencepiece", "protobuf", "datasets==4.3.0", "huggingface_hub>=0.34.0", "hf_transfer", "unsloth_zoo" "bitsandbytes", "accelerate", "peft", "trl", "triton"])
    pip_args.insert(0, "--no-deps")

pip_args = f"{" ".join(pip_args)}"
print(f"Pip args: {pip_args}")

Pip args: -qqq --extra-index-url https://download.pytorch.org/whl/cu130 torch==2.10.0 torchvision==0.25.0+cu130 torchaudio==2.10.0+cu130 unsloth transformers==4.56.2 tf_keras


In [3]:
%pip install {pip_args}

Note: you may need to restart the kernel to use updated packages.


### Global setup

This cell needs to run all the time (even when loading saved model)

In [ ]:
import re

random_state = 3407 # set to None to remove determinism
context_length = 2048
model_name = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"
# which layers to target with lora (recommended to target all linear layers, though i am not sure if these names are standardized, some models may have different names)
target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
saved_model_path = f"data/models/{re.sub(r"[^a-zA-Z0-9]+", "_", model_name)}_instruct_ft"

### Instruction fine-tuning setup

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = context_length,
    load_in_4bit = True,  # 4 bit quantization to reduce memory
)
# add lora adapters
model = FastModel.get_peft_model(
    model,
    r = 128,
    target_modules = target_modules,
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = random_state,
    use_rslora = False,
    loftq_config = None,
)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma3",
)

### Instruction dataset loading and preview

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset

instruct_ds_name = "codefuse-ai/Evol-instruction-66k"
# to reduce ds size set split = "train[:10000]" to use first 10k rows only
dataset = load_dataset(instruct_ds_name, split = "train")
assert(isinstance(dataset, Dataset))
dataset

### Instruction dataset processing

In [ ]:
# evol ds does not include system prompt, i set a static one for now
instruction = """
You are a helpful assistant that follows user's instructions
Be concise and factual and format all programming code you output using backticks or code blocks
Also do not forget to joke and put memes every now and then. Use super-chill and tone that matches user's vibes
Its OK to be unhinged sometimes
"""
def convert_to_chatml(example):
    return {
        "conversations": [
            {"role": "system", "content": instruction},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]}
        ]
    }
def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }
dataset = dataset.map(convert_to_chatml)
dataset = dataset.map(formatting_prompts_func, batched = True)
dataset[0]

### Instruction trainer setup

In [ ]:
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
from unsloth.chat_templates import train_on_responses_only

assert isinstance(dataset, Dataset)
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 100,
        learning_rate = 5e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

### Instruction fine-tuning

In [ ]:
trainer_stats = trainer.train() # set resume_from_checkpoint=True if continuing

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Inference test after instruction fine tuning

In [ ]:
from transformers import TextStreamer

sample_idx = 10
messages = [
    {'role': 'system','content':dataset['conversations'][sample_idx][0]['content']},
    {"role" : 'user', 'content' : dataset['conversations'][sample_idx][1]['content']}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
).removeprefix('<bos>')

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 125,
    # recommended settings for gemma3
    temperature = 1, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

### Saving instruction-tuned model

In [ ]:
model.save_pretrained(saved_model_path)  # Local saving
tokenizer.save_pretrained(saved_model_path)

## RL for tool calling

In [ ]:
# clear memory
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

### Loading instruction-tuned model

In [ ]:
# load saved model
from unsloth import FastLanguageModel

rl_rank = 64
rl_alpha = rl_rank * 2 # bigger -> greater gradients, could mean faster training

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = saved_model_path,
    max_seq_length = rl_context_length,
)

### Loading tool-calling dataset

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset

rl_dataset = "interstellarninja/hermes_reasoning_tool_use"
dataset = load_dataset(rl_dataset, split = "train")
dataset = dataset.to_pandas()[
    [
        "conversations", # combination of all below in a format selected by the author
        "tools", # available tools
        "task", # the human prompt (the first one)
        "category", # what type of task this is (can be used to formulate the system prompt)
        "source", # 'Salesforce-Xlam', 'Nvidia-When2Call', 'ToolAce', 'Nous-Hermes', 'Glaive', 'Xlam-Salesforce'
        "scenario_category" # 'single', 'multiturn', 'multistep', 'relevance'
    ]
]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\megak\Code\genai-labs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0224 21:44:28.890000 13928 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[tensorflow|WARNING]From c:\Users\megak\Code\genai-labs\.venv\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_colwidth', None)
dataset[dataset["scenario_category"] == "multiturn"].head(1)["conversations"].to_list()[0]

array([{'from': 'system', 'value': 'You are a deep thinking AI, you may use extremely long chains of thought to deeply consider the problem and deliberate with yourself via systematic reasoning processes to help come to a correct solution prior to answering. You should enclose your thoughts and internal monologue inside <think> </think> tags, and then provide your solution or response to the problem.\n\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. If available tools are not relevant in assisting with user query, just respond in natural conversational language. Don\'t make assumptions about what values to plug into functions. After calling & executing the functions, you will be provided with function results within <tool_response> </tool_response> XML tags. Here are the available tools:\n<tools>\n[{"name":"extract","description":"Extracts data from a Link

### Tool-calling dataset processing

Plan - keep it simple and at first just use the "conversations"

- Generate up to max_tokens or eos token and evaluate
- if from == "human" or "tool", append to context before continuing
- score based only on response, previous context should be discarded

Some ideas for scoring:

- Use vector similarity on outputted tokens (though this does not necessarily represent meaningful similarity)
- Use another model to judge whether output conveyed the same idea or not
- Prioritize correct use of tools - that should be higher score than generated thinking trace or output
- Most important is the final answer

In future, could try modifying the system prompt

In [ ]:
reasoning_start = "<think>"
reasoning_end   = "</think>"
tool_call_start = "<tool_call>"
tool_call_end = "</tool_call>"
tool_response_start = "<tool_response>"
tool_response_end = "</tool_response>"

# TODO: figure out how this chat template setup works so I can match to the dataset

chat_template = """
{% if messages[0]['role'] == 'system' %}
    {{ messages[0]['content'] + eos_token }}
    {% set loop_messages = messages[1:] %}
{% else %}
    {{ '{system_prompt}' + eos_token }}
    {% set loop_messages = messages %}
{% endif %}
{% for message in loop_messages %}
    {% if message['role'] == 'user' %}
        {{ message['content'] }}
    {% elif message['role'] == 'assistant' %}
        {{ message['content'] + eos_token }}
    {% endif %}
{% endfor %}
{% if add_generation_prompt %}{{ '{reasoning_start}' }}
{% endif %}
"""
# indentation in the chat template messes up training according to this guy: https://huggingface.co/DavidAU/How-To-Use-Reasoning-Thinking-Models-and-Create-Them
# and i don't have the time or compute to check if he is wrong
chat_template = "\n".join(line.lstrip() for line in chat_template.splitlines())
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

# truncate to half context length to reduce reasoning trace length
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)


### Tool-calling trainer setup

In [ ]:
from trl.trainer.grpo_trainer import GRPOTrainer
from trl.trainer.grpo_config import GRPOConfig
from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer.train()

In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)